# 🌾 AgriSmart AI — End-to-End Machine Learning Pipeline
> **International Competition Project** | Multi-Task Agricultural Intelligence System
>
> **Three AI Models in One Pipeline:**
> - 🌱 **Feature 1** — Fertilizer Recommendation
> - 🌾 **Feature 2** — Crop Recommendation  
> - ⚠️ **Feature 3** — Over-Fertilization Detection

---
**Author:** AgriSmart AI Team  
**Dataset:** 15,000 rows × 16 columns | Augmented from real agronomic datasets  
**Framework:** Scikit-learn + XGBoost + LightGBM + Ensemble Methods


## 📦 Section 1 — Library Installation & Imports

In [ ]:
# ── Install dependencies (run once) ─────────────────────────────────────────
# !pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn
# !pip install imbalanced-learn shap missingno optuna joblib

# ── Core Libraries ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import joblib
import os
import json
import time
warnings.filterwarnings('ignore')

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
import matplotlib.ticker as mticker

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
PALETTE = sns.color_palette('Set2')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans',
})

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
import lightgbm as lgb

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    ConfusionMatrixDisplay
)

# ── Optimization ──────────────────────────────────────────────────────────────
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# ── Imbalance Handling ────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Explainability ────────────────────────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed — skipping SHAP plots. Run: pip install shap')

print('✅ All libraries loaded successfully!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')
print(f'   XGBoost: {xgb.__version__}')
print(f'   LGBM   : {lgb.__version__}')
print(f'   sklearn: {__import__("sklearn").__version__}')

---
## 📂 Section 2 — Load Dataset

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────────
DATA_PATH = 'AgriSmart_Dataset.csv'   # ← adjust path if needed

df = pd.read_csv(DATA_PATH)

print('='*60)
print('       🌾  AgriSmart Dataset — Loaded Successfully')
print('='*60)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print(f'  Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print('='*60)
df.head()

In [ ]:
# ── Column Metadata ────────────────────────────────────────────────────────────
meta = {
    'Column': df.columns.tolist(),
    'DType': [str(df[c].dtype) for c in df.columns],
    'Non-Null': [df[c].notna().sum() for c in df.columns],
    'Null%': [round(df[c].isna().mean()*100, 2) for c in df.columns],
    'Unique': [df[c].nunique() for c in df.columns],
    'Sample': [str(df[c].iloc[0]) for c in df.columns],
}
pd.DataFrame(meta)

---
## 🔍 Section 3 — Data Exploration

In [ ]:
# ── Statistical Summary ───────────────────────────────────────────────────────
print('\n📊 Numerical Summary:')
df.describe().round(2)

In [ ]:
# ── Categorical Summary ───────────────────────────────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=np.number).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numerical columns   ({len(num_cols)}): {num_cols}\n')

for col in cat_cols:
    print(f'[{col}] → {df[col].nunique()} unique values')
    print(df[col].value_counts().head(8).to_string())
    print()

In [ ]:
# ── Target Distribution ───────────────────────────────────────────────────────
targets = ['Recommended_Fertilizer', 'Recommended_Crop', 'Overuse_Label']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
titles = ['🌱 Fertilizer Classes', '🌾 Crop Classes', '⚠️ Overuse Classes']

for ax, col, title in zip(axes, targets, titles):
    vc = df[col].value_counts()
    bars = ax.barh(vc.index[:15], vc.values[:15],
                   color=sns.color_palette('husl', len(vc[:15])))
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel('Count')
    for bar, val in zip(bars, vc.values[:15]):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=8)

plt.suptitle('Target Variable Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('target_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 🧹 Section 4 — Data Cleaning

In [ ]:
# ── Missing Value Audit ────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ Zero missing values — dataset is already clean!')
else:
    print(missing_df)

print(f'\nTotal Duplicates: {df.duplicated().sum()}')

In [ ]:
# ── Outlier Detection — IQR Method ────────────────────────────────────────────
def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower) | (data[col] > upper)]
    return len(outliers), lower, upper

print('📊 Outlier Report (IQR Method):')
print(f'{"Column":<20} {"Outliers":>10} {"Lower Bound":>14} {"Upper Bound":>14}')
print('-' * 60)
outlier_report = {}
for col in num_cols:
    cnt, lo, hi = detect_outliers_iqr(df, col)
    outlier_report[col] = (cnt, lo, hi)
    print(f'{col:<20} {cnt:>10,} {lo:>14.2f} {hi:>14.2f}')

In [ ]:
# ── Outlier Visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i >= len(axes): break
    ax = axes[i]
    bp = ax.boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[i % len(PALETTE)], alpha=0.7))
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Value')

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots — Outlier Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Outlier Treatment — Winsorize (cap at 1.5×IQR) ────────────────────────────
# We cap rather than remove to preserve all 15,000 rows

df_clean = df.copy()
WINSORIZE_COLS = ['Rainfall', 'Nitrogen', 'Phosphorus', 'Potassium', 'Land_Area']

for col in WINSORIZE_COLS:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lo = Q1 - 1.5 * IQR
    hi = Q3 + 1.5 * IQR
    before = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    df_clean[col] = df_clean[col].clip(lo, hi)
    print(f'  {col:<18}: {before:>5} outliers capped → [{lo:.2f}, {hi:.2f}]')

print(f'\n✅ Clean dataset shape: {df_clean.shape}')
print(f'   No rows removed — all 15,000 preserved!')

---
## 📊 Section 5 — Exploratory Data Analysis (EDA)
### 5.1 Univariate Analysis

In [ ]:
# ── Numerical Distributions — Histograms + KDE ────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(22, 14))
axes = axes.flatten()
colors = sns.color_palette('husl', len(num_cols))

for i, col in enumerate(num_cols):
    if i >= len(axes): break
    ax = axes[i]
    sns.histplot(df_clean[col], kde=True, ax=ax, color=colors[i], bins=40,
                 edgecolor='white', linewidth=0.3)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    mean_val = df_clean[col].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean={mean_val:.1f}')
    ax.legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Univariate Analysis — Numerical Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('univariate_numerical.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Skewness & Kurtosis Report ────────────────────────────────────────────────
skew_df = pd.DataFrame({
    'Skewness': df_clean[num_cols].skew().round(4),
    'Kurtosis': df_clean[num_cols].kurt().round(4),
})
skew_df['Skew_Status'] = skew_df['Skewness'].apply(
    lambda x: '⚠️ High Positive Skew' if x > 1 else ('⚠️ High Negative Skew' if x < -1 else '✅ Approx. Normal')
)
print('\n📐 Skewness & Kurtosis:')
skew_df

In [ ]:
# ── Categorical Univariate ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Soil Type
vc = df_clean['Soil_Type'].value_counts()
axes[0].pie(vc.values, labels=vc.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set3', len(vc)),
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Soil Type Distribution', fontweight='bold', fontsize=13)

# Overuse Label
vc2 = df_clean['Overuse_Label'].value_counts()
bars = axes[1].bar(vc2.index, vc2.values,
                   color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Over-Fertilization Label Distribution', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Count')
for bar, val in zip(bars, vc2.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}\n({val/len(df_clean)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Categorical Feature Univariate Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('univariate_categorical.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.2 Bivariate Analysis

In [ ]:
# ── NPK vs Overuse Label ──────────────────────────────────────────────────────
npk_cols = ['Nitrogen', 'Phosphorus', 'Potassium']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

overuse_order = ['Normal', 'Slight_Overuse', 'High_Overuse']
overuse_colors = {'Normal': '#2ecc71', 'Slight_Overuse': '#f39c12', 'High_Overuse': '#e74c3c'}

for ax, col in zip(axes, npk_cols):
    sns.boxplot(data=df_clean, x='Overuse_Label', y=col, order=overuse_order,
                palette=overuse_colors, ax=ax)
    ax.set_title(f'{col} vs Overuse Label', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Bivariate: NPK Levels by Overuse Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_npk_overuse.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Temperature & Humidity vs Top Crops ──────────────────────────────────────
top_crops = df_clean['Crop_Type'].value_counts().head(10).index.tolist()
df_top = df_clean[df_clean['Crop_Type'].isin(top_crops)]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.boxplot(data=df_top, x='Crop_Type', y='Temperature',
            palette='husl', ax=axes[0], order=top_crops)
axes[0].set_title('Temperature by Top 10 Crops', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_xlabel('')

sns.violinplot(data=df_top, x='Crop_Type', y='Humidity',
               palette='Set3', ax=axes[1], order=top_crops)
axes[1].set_title('Humidity Distribution by Top 10 Crops', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_xlabel('')

plt.suptitle('Bivariate: Climate Conditions per Crop', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_climate_crops.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Soil Type vs Crop Yield Factors ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Nitrogen', 'Soil_pH', 'Rainfall']):
    sns.stripplot(data=df_clean, x='Soil_Type', y=col, hue='Soil_Type',
                  palette='Set2', alpha=0.3, ax=ax, legend=False, size=2)
    sns.pointplot(data=df_clean, x='Soil_Type', y=col, color='black',
                  markers='D', linestyles='--', ax=ax, errorbar='sd')
    ax.set_title(f'{col} by Soil Type', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Bivariate: Soil Type vs Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_soil.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.3 Multivariate Analysis

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
corr = df_clean[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 9}, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — All Numerical Features', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('multivariate_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

# Print highly correlated pairs
print('\n🔥 High Correlations (|r| > 0.5):')
for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > 0.5:
            print(f'  {corr.columns[i]} ↔ {corr.columns[j]}: {corr.iloc[i,j]:.3f}')

In [ ]:
# ── Pairplot — NPK & Climate by Overuse ──────────────────────────────────────
pair_cols = ['Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Humidity']
df_pair = df_clean[pair_cols + ['Overuse_Label']].sample(1000, random_state=42)

g = sns.pairplot(df_pair, hue='Overuse_Label',
                 palette={'Normal': '#2ecc71', 'Slight_Overuse': '#f39c12', 'High_Overuse': '#e74c3c'},
                 plot_kws={'alpha': 0.4, 's': 20},
                 diag_kind='kde')
g.fig.suptitle('Multivariate Pairplot — NPK & Climate vs Overuse Label', y=1.02,
               fontsize=14, fontweight='bold')
plt.savefig('multivariate_pairplot.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Grouped Mean Heatmap — Crop × Feature ────────────────────────────────────
top15_crops = df_clean['Crop_Type'].value_counts().head(15).index
crop_feats = ['Temperature', 'Humidity', 'Rainfall', 'Soil_pH',
              'Nitrogen', 'Phosphorus', 'Potassium']

grouped = df_clean[df_clean['Crop_Type'].isin(top15_crops)].groupby('Crop_Type')[crop_feats].mean()
grouped_norm = (grouped - grouped.min()) / (grouped.max() - grouped.min())  # Normalize 0-1

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(grouped_norm, annot=grouped.round(1), fmt='.1f', cmap='YlOrRd',
            ax=ax, linewidths=0.4, cbar_kws={'label': 'Normalised Value'})
ax.set_title('Multivariate: Normalised Feature Means per Crop (Top 15)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('multivariate_crop_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 3D Scatter: N, P, K coloured by Overuse ───────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

sample = df_clean.sample(2000, random_state=42)
color_map = {'Normal': '#2ecc71', 'Slight_Overuse': '#f39c12', 'High_Overuse': '#e74c3c'}
colors_arr = sample['Overuse_Label'].map(color_map)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

for label, color in color_map.items():
    mask = sample['Overuse_Label'] == label
    ax.scatter(sample[mask]['Nitrogen'], sample[mask]['Phosphorus'],
               sample[mask]['Potassium'], c=color, label=label, alpha=0.5, s=15)

ax.set_xlabel('Nitrogen (N)', fontsize=10)
ax.set_ylabel('Phosphorus (P)', fontsize=10)
ax.set_zlabel('Potassium (K)', fontsize=10)
ax.set_title('3D NPK Space — Coloured by Overuse Label', fontsize=13, fontweight='bold')
ax.legend()
plt.savefig('multivariate_3d_npk.png', bbox_inches='tight', dpi=150)
plt.show()

---
## ⚙️ Section 6 — Data Preprocessing

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df_ml = df_clean.copy()

# Ratio features (agronomically meaningful)
df_ml['N_P_ratio']      = df_ml['Nitrogen']   / (df_ml['Phosphorus']  + 1e-6)
df_ml['N_K_ratio']      = df_ml['Nitrogen']   / (df_ml['Potassium']   + 1e-6)
df_ml['P_K_ratio']      = df_ml['Phosphorus'] / (df_ml['Potassium']   + 1e-6)
df_ml['NPK_sum']        = df_ml['Nitrogen'] + df_ml['Phosphorus'] + df_ml['Potassium']
df_ml['NPK_balance']    = df_ml['NPK_sum']    / (df_ml['Land_Area']   + 1e-6)  # NPK density
df_ml['Heat_Index']     = df_ml['Temperature'] * df_ml['Humidity'] / 100       # apparent heat
df_ml['Water_Stress']   = df_ml['Rainfall']   / (df_ml['Temperature'] + 1)    # water-temp ratio
df_ml['N_excess']       = (df_ml['Nitrogen']   - df_ml['Recommended_N']).clip(lower=0)
df_ml['P_excess']       = (df_ml['Phosphorus'] - df_ml['Recommended_P']).clip(lower=0)
df_ml['K_excess']       = (df_ml['Potassium']  - df_ml['Recommended_K']).clip(lower=0)
df_ml['Total_excess']   = df_ml['N_excess'] + df_ml['P_excess'] + df_ml['K_excess']

print(f'✅ Feature engineering complete!')
print(f'   Original features : {df_clean.shape[1]}')
print(f'   Engineered features added: 11')
print(f'   Total features    : {df_ml.shape[1]}')

In [ ]:
# ── Label Encoding for Targets ─────────────────────────────────────────────────
le_fert  = LabelEncoder()
le_crop  = LabelEncoder()
le_over  = LabelEncoder()
le_soil  = LabelEncoder()

df_ml['Soil_Type_enc']             = le_soil.fit_transform(df_ml['Soil_Type'])
df_ml['Target_Fertilizer']         = le_fert.fit_transform(df_ml['Recommended_Fertilizer'])
df_ml['Target_Crop']               = le_crop.fit_transform(df_ml['Recommended_Crop'])
df_ml['Target_Overuse']            = le_over.fit_transform(df_ml['Overuse_Label'])

# Save encoders
os.makedirs('models', exist_ok=True)
joblib.dump(le_fert, 'models/le_fertilizer.pkl')
joblib.dump(le_crop,  'models/le_crop.pkl')
joblib.dump(le_over,  'models/le_overuse.pkl')
joblib.dump(le_soil,  'models/le_soil.pkl')

print('Label Encoder Classes:')
print(f'  Fertilizer ({len(le_fert.classes_)} classes): {list(le_fert.classes_)}')
print(f'  Crop       ({len(le_crop.classes_)} classes): {list(le_crop.classes_)[:8]}...')
print(f'  Overuse    ({len(le_over.classes_)} classes): {list(le_over.classes_)}')

In [ ]:
# ── Feature Selection ──────────────────────────────────────────────────────────
# Input features (exclude raw categorical targets and their encoded targets)
FEATURE_COLS = [
    'Temperature', 'Humidity', 'Rainfall', 'Soil_Type_enc', 'Soil_pH',
    'Nitrogen', 'Phosphorus', 'Potassium', 'Land_Area',
    'Recommended_N', 'Recommended_P', 'Recommended_K',
    'N_P_ratio', 'N_K_ratio', 'P_K_ratio', 'NPK_sum', 'NPK_balance',
    'Heat_Index', 'Water_Stress', 'N_excess', 'P_excess', 'K_excess', 'Total_excess'
]

X = df_ml[FEATURE_COLS]
y_fert  = df_ml['Target_Fertilizer']
y_crop  = df_ml['Target_Crop']
y_over  = df_ml['Target_Overuse']

print(f'Feature matrix: {X.shape}')
print(f'Target - Fertilizer : {y_fert.nunique()} classes')
print(f'Target - Crop       : {y_crop.nunique()} classes')
print(f'Target - Overuse    : {y_over.nunique()} classes')
print(f'Null in X: {X.isnull().sum().sum()}')

In [ ]:
# ── Train / Validation / Test Split (70/15/15) ────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.15
VAL_SIZE     = 0.15 / 0.85   # 15% of remaining after test split

splits = {}
for name, y in [('fertilizer', y_fert), ('crop', y_crop), ('overuse', y_over)]:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_temp)
    splits[name] = (X_train, X_val, X_test, y_train, y_val, y_test)
    print(f'[{name:12}] Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# ── Feature Scaling ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(splits['fertilizer'][0])
X_val_sc   = scaler.transform(splits['fertilizer'][1])
X_test_sc  = scaler.transform(splits['fertilizer'][2])

joblib.dump(scaler, 'models/scaler.pkl')
print('\n✅ Scaler fitted and saved.')

---
## 🤖 Section 7 — Model Selection & Training
> We train **7 algorithms** on all 3 tasks and select the best per task.

In [ ]:
# ── Model Zoo ─────────────────────────────────────────────────────────────────
MODEL_ZOO = {
    'Decision Tree'       : DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150, random_state=RANDOM_STATE),
    'XGBoost'             : xgb.XGBClassifier(n_estimators=200, use_label_encoder=False,
                                              eval_metric='mlogloss', random_state=RANDOM_STATE,
                                              tree_method='hist', n_jobs=-1),
    'LightGBM'            : lgb.LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                               n_jobs=-1, verbose=-1),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

print(f'Model Zoo: {len(MODEL_ZOO)} algorithms ready')
for m in MODEL_ZOO: print(f'  • {m}')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
import time

all_results = {}  # task → {model_name: {acc, f1, val_acc, train_time}}

TASK_NAMES = {
    'fertilizer': ('Fertilizer Recommendation', le_fert),
    'crop'      : ('Crop Recommendation',        le_crop),
    'overuse'   : ('Over-Fertilization Check',   le_over),
}

for task, (task_label, le) in TASK_NAMES.items():
    X_train, X_val, X_test, y_train, y_val, y_test = splits[task]
    all_results[task] = {}
    
    print(f'\n{"="*65}')
    print(f'  🎯 Task: {task_label}')
    print(f'{"="*65}')
    print(f'{"Model":<22} {"Train Acc":>10} {"Val Acc":>10} {"Val F1":>10} {"Time(s)":>9}')
    print('-'*65)
    
    for name, model in MODEL_ZOO.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - t0

        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_preds = model.predict(X_val)
        val_acc   = accuracy_score(y_val, val_preds)
        val_f1    = f1_score(y_val, val_preds, average='weighted')

        all_results[task][name] = {
            'model': model,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'val_f1': val_f1,
            'time': elapsed,
            'X_test': X_test, 'y_test': y_test
        }
        
        overfitting = '⚠️' if (train_acc - val_acc) > 0.05 else ''
        print(f'{name:<22} {train_acc:>10.4f} {val_acc:>10.4f} {val_f1:>10.4f} {elapsed:>9.2f} {overfitting}')

print('\n✅ All models trained!')

---
## 📈 Section 8 — Model Evaluation

In [ ]:
# ── Comparative Accuracy Chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, task in zip(axes, ['fertilizer', 'crop', 'overuse']):
    results = all_results[task]
    names  = list(results.keys())
    val_accs = [results[n]['val_acc'] for n in names]
    val_f1s  = [results[n]['val_f1']  for n in names]
    
    x = np.arange(len(names))
    w = 0.35
    bars1 = ax.bar(x - w/2, val_accs, w, label='Val Accuracy', color='#3498db', alpha=0.85)
    bars2 = ax.bar(x + w/2, val_f1s,  w, label='Val F1-Score', color='#e67e22', alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=40, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title(TASK_NAMES[task][0], fontweight='bold', fontsize=12)
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    ax.axhline(0.9, color='green', linestyle='--', linewidth=1, alpha=0.5, label='90% threshold')
    
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
                ha='center', fontsize=7, rotation=90)

plt.suptitle('Model Comparison — Validation Accuracy & F1-Score', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Best Models Selection ─────────────────────────────────────────────────────
best_models = {}

for task in ['fertilizer', 'crop', 'overuse']:
    best_name = max(all_results[task], key=lambda n: all_results[task][n]['val_f1'])
    best_models[task] = best_name
    info = all_results[task][best_name]
    print(f'[{TASK_NAMES[task][0]}]')
    print(f'   Best Model : {best_name}')
    print(f'   Val Acc    : {info["val_acc"]:.4f}')
    print(f'   Val F1     : {info["val_f1"]:.4f}')
    print()

In [ ]:
# ── Confusion Matrices on TEST SET ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, task in zip(axes, ['fertilizer', 'crop', 'overuse']):
    best_name = best_models[task]
    result    = all_results[task][best_name]
    model     = result['model']
    le        = TASK_NAMES[task][1]
    X_test_t  = result['X_test']
    y_test_t  = result['y_test']
    
    y_pred = model.predict(X_test_t)
    cm = confusion_matrix(y_test_t, y_pred)
    
    if len(le.classes_) <= 12:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=le.classes_, yticklabels=le.classes_,
                    ax=ax, cbar=False, linewidths=0.3)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', rotation=0,  labelsize=7)
    else:
        # Too many classes — show normalised
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        sns.heatmap(cm_norm, cmap='Blues', ax=ax, cbar=True)
        ax.set_title(f'{TASK_NAMES[task][0]}\n(Normalised — {len(le.classes_)} classes)',
                     fontweight='bold', fontsize=11)
        continue
    
    test_acc = accuracy_score(y_test_t, y_pred)
    ax.set_title(f'{TASK_NAMES[task][0]}\n{best_name} | Test Acc: {test_acc:.4f}',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Best Models on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Full Classification Report ────────────────────────────────────────────────
for task in ['fertilizer', 'crop', 'overuse']:
    best_name = best_models[task]
    result    = all_results[task][best_name]
    le        = TASK_NAMES[task][1]
    y_pred    = result['model'].predict(result['X_test'])
    
    print(f'\n{"="*65}')
    print(f'  Task: {TASK_NAMES[task][0]} | Model: {best_name}')
    print(f'{"="*65}')
    print(classification_report(result['y_test'], y_pred, target_names=le.classes_))

In [ ]:
# ── Cross-Validation on Best Models ──────────────────────────────────────────
print('📊 5-Fold Stratified Cross-Validation:')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for task in ['fertilizer', 'crop', 'overuse']:
    best_name = best_models[task]
    model     = all_results[task][best_name]['model']
    X_tr      = splits[task][0]
    y_tr      = splits[task][3]
    
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_weighted', n_jobs=-1)
    print(f'\n  [{TASK_NAMES[task][0]}]')
    print(f'  Model : {best_name}')
    print(f'  Fold F1 Scores : {cv_scores.round(4)}')
    print(f'  Mean ± Std     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## 🔧 Section 9 — Model Optimization (Hyperparameter Tuning)

In [ ]:
# ── Hyperparameter Tuning — RandomizedSearchCV on Best Models ─────────────────
# We tune one representative model per task for competition-grade performance.

PARAM_GRIDS = {
    'XGBoost': {
        'n_estimators'    : [100, 200, 300, 500],
        'max_depth'       : [4, 6, 8, 10],
        'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
        'subsample'       : [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
        'min_child_weight': [1, 3, 5],
        'gamma'           : [0, 0.1, 0.2],
        'reg_alpha'       : [0, 0.01, 0.1],
        'reg_lambda'      : [1, 1.5, 2.0],
    },
    'LightGBM': {
        'n_estimators'    : [100, 200, 300, 500],
        'max_depth'       : [4, 6, 8, -1],
        'learning_rate'   : [0.01, 0.05, 0.1],
        'num_leaves'      : [31, 63, 127],
        'subsample'       : [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
        'min_child_samples': [10, 20, 50],
        'reg_alpha'       : [0, 0.01, 0.1],
        'reg_lambda'      : [0, 0.01, 0.1],
    },
    'Random Forest': {
        'n_estimators'    : [200, 300, 500],
        'max_depth'       : [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features'    : ['sqrt', 'log2', 0.5],
    }
}

tuned_models = {}

for task in ['fertilizer', 'crop', 'overuse']:
    best_name   = best_models[task]
    base_model  = MODEL_ZOO[best_name].__class__  # fresh instance
    param_grid  = PARAM_GRIDS.get(best_name, None)
    X_tr, _, _, y_tr, _, _ = splits[task]
    
    print(f'\n🔧 Tuning [{TASK_NAMES[task][0]}] → {best_name}')
    
    if param_grid is None:
        print(f'  No param grid for {best_name} — using default.')
        tuned_models[task] = all_results[task][best_name]['model']
        continue
    
    # Clone the model
    if best_name == 'XGBoost':
        base = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',
                                  random_state=RANDOM_STATE, tree_method='hist', n_jobs=-1)
    elif best_name == 'LightGBM':
        base = lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    elif best_name == 'Random Forest':
        base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
    else:
        tuned_models[task] = all_results[task][best_name]['model']
        continue
    
    rs = RandomizedSearchCV(
        base, param_grid, n_iter=30, cv=3,
        scoring='f1_weighted', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=0
    )
    t0 = time.time()
    rs.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    
    tuned_models[task] = rs.best_estimator_
    print(f'  ✅ Best CV F1 : {rs.best_score_:.4f} (searched in {elapsed:.1f}s)')
    print(f'  Best Params  : {rs.best_params_}')

print('\n✅ Hyperparameter tuning complete!')

---
## 📊 Section 10 — Post-Optimization Evaluation

In [ ]:
# ── Tuned Model Performance on Test Set ──────────────────────────────────────
print('📊 Tuned Model — Final Test Set Performance:')
print(f'{"Task":<32} {"Test Acc":>10} {"Test F1":>10} {"Precision":>10} {"Recall":>10}')
print('─' * 76)

final_scores = {}
for task in ['fertilizer', 'crop', 'overuse']:
    model  = tuned_models[task]
    X_test = splits[task][2]
    y_test = splits[task][5]
    
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='weighted')
    prec   = precision_score(y_test, y_pred, average='weighted')
    rec    = recall_score(y_test, y_pred, average='weighted')
    
    final_scores[task] = {'acc': acc, 'f1': f1, 'prec': prec, 'rec': rec}
    label = TASK_NAMES[task][0]
    print(f'{label:<32} {acc:>10.4f} {f1:>10.4f} {prec:>10.4f} {rec:>10.4f}')

In [ ]:
# ── Before vs After Tuning Comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

tasks_labels = [TASK_NAMES[t][0] for t in ['fertilizer', 'crop', 'overuse']]
before_f1 = [all_results[t][best_models[t]]['val_f1'] for t in ['fertilizer', 'crop', 'overuse']]
after_f1  = [final_scores[t]['f1'] for t in ['fertilizer', 'crop', 'overuse']]

x = np.arange(3)
w = 0.35
b1 = ax.bar(x - w/2, before_f1, w, label='Before Tuning (Val F1)', color='#95a5a6', alpha=0.85)
b2 = ax.bar(x + w/2, after_f1,  w, label='After Tuning (Test F1)', color='#2ecc71', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(tasks_labels, fontsize=12)
ax.set_ylim(0.7, 1.05)
ax.set_ylabel('F1 Score (Weighted)', fontsize=12)
ax.set_title('Before vs After Hyperparameter Tuning', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(1.0, color='black', linestyle=':', alpha=0.3)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f'{h:.4f}',
            ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('tuning_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Feature Importance — All Three Tasks ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for ax, task in zip(axes, ['fertilizer', 'crop', 'overuse']):
    model = tuned_models[task]
    
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1][:15]
        top_features = [FEATURE_COLS[i] for i in indices]
        top_imp      = importances[indices]
        
        colors_fi = sns.color_palette('RdYlGn', len(top_features))
        bars = ax.barh(range(len(top_features)), top_imp[::-1],
                       color=colors_fi, edgecolor='white')
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features[::-1], fontsize=9)
        ax.set_title(f'{TASK_NAMES[task][0]}\nTop 15 Feature Importances', fontweight='bold')
        ax.set_xlabel('Importance')
    else:
        ax.text(0.5, 0.5, f'Feature importance\nnot available\nfor this model',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title(TASK_NAMES[task][0], fontweight='bold')

plt.suptitle('Feature Importance — Tuned Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importances.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── SHAP Explainability (if available) ───────────────────────────────────────
if SHAP_AVAILABLE:
    print('🔍 Generating SHAP explanations...')
    for task in ['overuse']:   # Run on overuse (smaller classes, faster)
        model  = tuned_models[task]
        X_test = splits[task][2]
        
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        
        plt.figure(figsize=(12, 7))
        if isinstance(shap_values, list):
            shap.summary_plot(shap_values[0], X_test,
                              feature_names=FEATURE_COLS, show=False)
        else:
            shap.summary_plot(shap_values, X_test,
                              feature_names=FEATURE_COLS, show=False)
        plt.title(f'SHAP Summary — {TASK_NAMES[task][0]}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_{task}.png', bbox_inches='tight', dpi=150)
        plt.show()
else:
    print('⚠️ SHAP not installed. Run: pip install shap')

---
## 🏆 Section 11 — Ensemble Model (Stacking)

In [ ]:
# ── Stacking Ensemble — Competition-Grade Final Models ─────────────────────────
# Stacking combines predictions from multiple base learners using a meta-learner.

ensemble_models = {}

for task in ['fertilizer', 'crop', 'overuse']:
    print(f'\n🏗️  Building Stacking Ensemble for [{TASK_NAMES[task][0]}]...')
    
    X_tr, X_val, X_test, y_tr, y_val, y_test = splits[task]
    X_tr_full = np.vstack([X_tr, X_val])   # Use train+val for final ensemble
    y_tr_full = np.concatenate([y_tr, y_val])
    
    base_learners = [
        ('rf',   RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
        ('xgb',  xgb.XGBClassifier(n_estimators=200, use_label_encoder=False,
                                    eval_metric='mlogloss', tree_method='hist',
                                    random_state=RANDOM_STATE, n_jobs=-1)),
        ('lgbm', lgb.LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                     n_jobs=-1, verbose=-1)),
        ('et',   ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    ]
    meta_learner = LogisticRegression(max_iter=500, multi_class='multinomial',
                                      solver='lbfgs', random_state=RANDOM_STATE)
    
    stacking = StackingClassifier(
        estimators=base_learners,
        final_estimator=meta_learner,
        cv=3,
        n_jobs=-1,
        passthrough=True   # include original features in meta-learner
    )
    
    t0 = time.time()
    stacking.fit(X_tr_full, y_tr_full)
    elapsed = time.time() - t0
    
    y_pred = stacking.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    
    ensemble_models[task] = stacking
    print(f'  ✅ Test Acc : {acc:.4f} | Test F1 : {f1:.4f} | Time: {elapsed:.1f}s')
    
    # Compare with tuned model
    prev_f1 = final_scores[task]['f1']
    delta   = f1 - prev_f1
    symbol  = '📈' if delta > 0 else '📉'
    print(f'  {symbol} vs. Tuned Model: {delta:+.4f}')

print('\n✅ All ensemble models built!')

In [ ]:
# ── Final Scorecard ───────────────────────────────────────────────────────────
print('\n' + '='*70)
print('  🏆  AGRISMART AI — FINAL MODEL SCORECARD')
print('='*70)
print(f'{"Task":<30} {"Model":<20} {"Test Acc":>10} {"Test F1":>10}')
print('─'*70)

for task in ['fertilizer', 'crop', 'overuse']:
    for label, model in [('Stacking', ensemble_models[task]),
                          ('Tuned',   tuned_models[task])]:
        X_test = splits[task][2]
        y_test = splits[task][5]
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='weighted')
        task_label = TASK_NAMES[task][0]
        star = '⭐' if label == 'Stacking' else '  '
        print(f'{star} {task_label:<28} {label:<20} {acc:>10.4f} {f1:>10.4f}')
    print()

print('='*70)
print('  ⭐ = Recommended final model for deployment')
print('='*70)

---
## 💾 Section 12 — Model Saving

In [ ]:
# ── Save All Models & Artifacts ───────────────────────────────────────────────
os.makedirs('models', exist_ok=True)

# Save ensemble (final production models)
joblib.dump(ensemble_models['fertilizer'], 'models/model_fertilizer.pkl')
joblib.dump(ensemble_models['crop'],       'models/model_crop.pkl')
joblib.dump(ensemble_models['overuse'],    'models/model_overuse.pkl')

# Save tuned models (backup)
joblib.dump(tuned_models['fertilizer'],    'models/tuned_fertilizer.pkl')
joblib.dump(tuned_models['crop'],          'models/tuned_crop.pkl')
joblib.dump(tuned_models['overuse'],       'models/tuned_overuse.pkl')

# Save preprocessing artifacts
joblib.dump(scaler,   'models/scaler.pkl')
joblib.dump(le_fert,  'models/le_fertilizer.pkl')
joblib.dump(le_crop,  'models/le_crop.pkl')
joblib.dump(le_over,  'models/le_overuse.pkl')
joblib.dump(le_soil,  'models/le_soil.pkl')

# Save feature list and metadata
metadata = {
    'feature_cols'     : FEATURE_COLS,
    'fertilizer_classes': list(le_fert.classes_),
    'crop_classes'     : list(le_crop.classes_),
    'overuse_classes'  : list(le_over.classes_),
    'soil_classes'     : list(le_soil.classes_),
    'final_scores'     : {
        task: {k: round(v, 4) for k, v in final_scores[task].items()}
        for task in final_scores
    },
    'best_base_models' : best_models,
}

with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Verify all files saved
print('\n✅ All files saved in /models/:')
for fname in sorted(os.listdir('models')):
    fpath = os.path.join('models', fname)
    size  = os.path.getsize(fpath)
    print(f'  {fname:<35} {size/1024:>8.1f} KB')

---
## 🧪 Section 13 — Sample Model Test Code

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  🧪 SAMPLE TEST CODE — Copy & Run This Cell to Test the Deployed Models
# ═══════════════════════════════════════════════════════════════════════════════

import joblib, json, numpy as np, pandas as pd

# ── Load Saved Artifacts ──────────────────────────────────────────────────────
model_fert = joblib.load('models/model_fertilizer.pkl')
model_crop = joblib.load('models/model_crop.pkl')
model_over = joblib.load('models/model_overuse.pkl')
scaler     = joblib.load('models/scaler.pkl')
le_fert    = joblib.load('models/le_fertilizer.pkl')
le_crop    = joblib.load('models/le_crop.pkl')
le_over    = joblib.load('models/le_overuse.pkl')
le_soil    = joblib.load('models/le_soil.pkl')

with open('models/metadata.json') as f:
    meta = json.load(f)

FEATURE_COLS = meta['feature_cols']

# ── Prediction Helper Function ────────────────────────────────────────────────
def predict_agrismart(
    temperature, humidity, rainfall, soil_type,
    soil_ph, nitrogen, phosphorus, potassium,
    land_area, recommended_n, recommended_p, recommended_k
):
    """
    AgriSmart AI — Full Prediction Pipeline

    Parameters
    ----------
    temperature   : float  — °C
    humidity      : float  — %
    rainfall      : float  — mm
    soil_type     : str    — 'Sandy' | 'Clay' | 'Loamy'
    soil_ph       : float  — 4.0 to 9.0
    nitrogen      : float  — kg/ha (actual soil N)
    phosphorus    : float  — kg/ha (actual soil P)
    potassium     : float  — kg/ha (actual soil K)
    land_area     : float  — acres
    recommended_n : float  — ideal N for your crop
    recommended_p : float  — ideal P for your crop
    recommended_k : float  — ideal K for your crop

    Returns
    -------
    dict with recommended_crop, recommended_fertilizer, overuse_status, confidence
    """
    # Encode soil type
    soil_enc = le_soil.transform([soil_type])[0]

    # Feature engineering (must match training)
    N_P_ratio    = nitrogen    / (phosphorus  + 1e-6)
    N_K_ratio    = nitrogen    / (potassium   + 1e-6)
    P_K_ratio    = phosphorus  / (potassium   + 1e-6)
    NPK_sum      = nitrogen + phosphorus + potassium
    NPK_balance  = NPK_sum     / (land_area   + 1e-6)
    Heat_Index   = temperature * humidity / 100
    Water_Stress = rainfall    / (temperature + 1)
    N_excess     = max(0, nitrogen   - recommended_n)
    P_excess     = max(0, phosphorus - recommended_p)
    K_excess     = max(0, potassium  - recommended_k)
    Total_excess = N_excess + P_excess + K_excess

    row = [
        temperature, humidity, rainfall, soil_enc, soil_ph,
        nitrogen, phosphorus, potassium, land_area,
        recommended_n, recommended_p, recommended_k,
        N_P_ratio, N_K_ratio, P_K_ratio, NPK_sum, NPK_balance,
        Heat_Index, Water_Stress, N_excess, P_excess, K_excess, Total_excess
    ]

    X_input = np.array(row).reshape(1, -1)
    X_scaled = scaler.transform(X_input)

    # ── Predictions ──────────────────────────────────────────────────────────
    pred_crop  = le_crop.inverse_transform(model_crop.predict(X_scaled))[0]
    pred_fert  = le_fert.inverse_transform(model_fert.predict(X_scaled))[0]
    pred_over  = le_over.inverse_transform(model_over.predict(X_scaled))[0]

    # Confidence scores (if model supports predict_proba)
    try:
        conf_crop = round(model_crop.predict_proba(X_scaled).max() * 100, 2)
        conf_fert = round(model_fert.predict_proba(X_scaled).max() * 100, 2)
        conf_over = round(model_over.predict_proba(X_scaled).max() * 100, 2)
    except:
        conf_crop = conf_fert = conf_over = 'N/A'

    return {
        'Recommended Crop'       : pred_crop,
        'Crop Confidence'        : f'{conf_crop}%',
        'Recommended Fertilizer' : pred_fert,
        'Fertilizer Confidence'  : f'{conf_fert}%',
        'Overuse Status'         : pred_over,
        'Overuse Confidence'     : f'{conf_over}%',
    }

print('✅ predict_agrismart() function loaded. See examples below.')

In [ ]:
# ── Example 1: Rice-growing field, slightly over-fertilized ──────────────────
result1 = predict_agrismart(
    temperature   = 28.0,
    humidity      = 75.0,
    rainfall      = 220.0,
    soil_type     = 'Clay',
    soil_ph       = 6.5,
    nitrogen      = 100.0,   # Actual N (ideal is ~80)
    phosphorus    = 42.0,    # Actual P (ideal is ~40)
    potassium     = 38.0,    # Actual K (ideal is ~40)
    land_area     = 5.0,
    recommended_n = 80,
    recommended_p = 40,
    recommended_k = 40
)

print('\n🌾 Example 1: Rice Field (Clay Soil, High Rainfall)')
print('─' * 50)
for k, v in result1.items():
    print(f'  {k:<28}: {v}')

In [ ]:
# ── Example 2: Wheat field, dry region, balanced NPK ─────────────────────────
result2 = predict_agrismart(
    temperature   = 22.0,
    humidity      = 55.0,
    rainfall      = 85.0,
    soil_type     = 'Loamy',
    soil_ph       = 7.2,
    nitrogen      = 115.0,
    phosphorus    = 58.0,
    potassium     = 39.0,
    land_area     = 10.0,
    recommended_n = 120,
    recommended_p = 60,
    recommended_k = 40
)

print('\n🌾 Example 2: Wheat Field (Loamy Soil, Dry Region)')
print('─' * 50)
for k, v in result2.items():
    print(f'  {k:<28}: {v}')

In [ ]:
# ── Example 3: Sandy soil, tropical, heavy overuse ───────────────────────────
result3 = predict_agrismart(
    temperature   = 35.0,
    humidity      = 85.0,
    rainfall      = 400.0,
    soil_type     = 'Sandy',
    soil_ph       = 5.8,
    nitrogen      = 250.0,   # Way above ideal
    phosphorus    = 90.0,    # Way above ideal
    potassium     = 150.0,   # Way above ideal
    land_area     = 3.0,
    recommended_n = 80,
    recommended_p = 40,
    recommended_k = 40
)

print('\n⚠️  Example 3: Tropical Sandy Soil — Heavy Overuse')
print('─' * 50)
for k, v in result3.items():
    print(f'  {k:<28}: {v}')

In [ ]:
# ── Batch Prediction Example ──────────────────────────────────────────────────
batch_inputs = [
    dict(temperature=28.0, humidity=75.0, rainfall=220.0, soil_type='Clay',
         soil_ph=6.5, nitrogen=100.0, phosphorus=42.0, potassium=38.0,
         land_area=5.0, recommended_n=80, recommended_p=40, recommended_k=40),
    dict(temperature=22.0, humidity=55.0, rainfall=85.0,  soil_type='Loamy',
         soil_ph=7.2, nitrogen=115.0, phosphorus=58.0, potassium=39.0,
         land_area=10.0, recommended_n=120, recommended_p=60, recommended_k=40),
    dict(temperature=35.0, humidity=85.0, rainfall=400.0, soil_type='Sandy',
         soil_ph=5.8, nitrogen=250.0, phosphorus=90.0, potassium=150.0,
         land_area=3.0, recommended_n=80, recommended_p=40, recommended_k=40),
]

batch_results = [predict_agrismart(**inp) for inp in batch_inputs]
batch_df = pd.DataFrame(batch_results)
batch_df.index = ['Farm A', 'Farm B', 'Farm C']

print('\n📋 Batch Prediction Results:')
batch_df

---
## 📋 Section 14 — Final Summary Report

In [ ]:
# ── Competition-Ready Summary ─────────────────────────────────────────────────
print('\n' + '█'*70)
print('█' + ' '*68 + '█')
print('█' + '   🌾  AgriSmart AI — Competition Summary Report'.center(68) + '█')
print('█' + ' '*68 + '█')
print('█'*70)

print('''
DATASET
  ● Rows          : 15,000  |  Columns: 16
  ● Crops         : 41 varieties
  ● Fertilizers   : 24 types
  ● Augmentation  : Gaussian noise + domain-constraint clipping

FEATURE ENGINEERING
  ● 11 derived features added (NPK ratios, heat index, excess scores, etc.)
  ● Total input features per model: 23

MODEL ARCHITECTURE
  ● 3 independent classification pipelines (one per AI feature)
  ● 7 base algorithms benchmarked per task
  ● Hyperparameter tuning: RandomizedSearchCV (30 iterations, 3-fold CV)
  ● Final models: Stacking Ensemble (RF + XGBoost + LightGBM + ExtraTrees)
  ● Meta-learner: Multinomial Logistic Regression with passthrough

EVALUATION PROTOCOL
  ● Train / Val / Test split: 70 / 15 / 15
  ● Stratified K-Fold CV: 5 folds
  ● Metrics: Accuracy, Weighted F1, Precision, Recall

EXPLAINABILITY
  ● Feature importance plots (all 3 models)
  ● SHAP TreeExplainer (if SHAP installed)

SAVED ARTIFACTS
  ● models/model_fertilizer.pkl     — Production: Fertilizer Recommender
  ● models/model_crop.pkl           — Production: Crop Recommender
  ● models/model_overuse.pkl        — Production: Overuse Detector
  ● models/scaler.pkl               — StandardScaler
  ● models/le_*.pkl                 — Label Encoders (4 files)
  ● models/metadata.json            — Classes, features, scores
''')
print('█'*70)